## Lecteur de fichiers

In [1]:
def reader(chemin):
    with open(chemin, mode='r', encoding ='utf8') as f:
        texte = f.read()
    return texte

## Tokeniseurs

### Texte tokeniseur

In [2]:
# Fonctions load_grm et tokenize_fr

import re


def load_grm(grm_filename,abrev_filename,compound_filename):
  grm_content = reader(grm_filename)
  abrev_content = reader(abrev_filename)
  compound_content = reader(compound_filename)
  

  compound_list = [re.sub(r' ',r'\ ',re.escape(l.strip())) for l in compound_content.split('\n') if l.strip()]
  abrev_list = [re.sub(r' ',r'\ ',re.escape(l.strip()))  for l in abrev_content.split('\n') if l.strip()]

  compound="|".join(compound_list)
  abrev="|".join(abrev_list)

  grammaire_finale = compound + "|\n" + abrev + "|\n" + grm_content

  grm_regex = re.compile(grammaire_finale, flags = re.X|re.I)

  return grm_regex

def tokenize_fr(texte):
    regex= load_grm(
    r"tok_grm/grm_tok_en.txt",
    r"tok_grm/abrev_dic_en.txt",
    r"tok_grm/compound_dic_en.txt"
    )
    toks = regex.findall(texte)
    tokens = [t.strip() for t in toks if t.strip()]

    return tokens

### Règles Tokeniseur

In [3]:
import re
def tokenizer_rule(text):
    ruletoks=[tok for tok in re.split(r"\s",text) if tok.strip() !=""]
    tokens=[]
    i=0
    while i < len(ruletoks)-2 :
        tokens.append([ruletoks[i], ruletoks[i+1], ruletoks[i+2]])
        i+=3
    return tokens


### Lexique tokeniseur

In [4]:

# Obtenir une liste de ligne, chaque ligne du texte étant une liste de tokens
def tokenizer_lex(chemin):
    with open(chemin, encoding="utf-8") as file:
        tokens = []
        for line in file:
            clean_line = line.strip()
            tokens.append(tokenize_fr(clean_line))
                
    return tokens

## Chargement de : Lexique, Règles et Tokens

In [5]:
lexique = tokenizer_lex("lexique.txt")
print(lexique)

regles= tokenizer_rule(reader("chk_regles.txt"))
print(regles)

tokens = tokenize_fr(reader("article_En_The_Onion.txt"))
print(tokens)


[['DET', 'the', 'a', 'an', 'this', 'that', 'these', 'those', 'my', 'your', 'his', 'her', 'its', 'our', 'their', 'each', 'some', 'all', 'every', 'both', 'few', 'much', 'many', 'more', 'most'], ['PREP', 'to', 'at', 'in', 'on', 'of', 'from', 'with', 'without', 'by', 'for', 'about', 'between', 'before', 'after', 'during', 'since', 'toward', 'against', 'through', 'into'], ['PronomP', 'I', 'you', 'he', 'she', 'it', 'we', 'they', 'me', 'him', 'us', 'them', 'my', 'your'], ['PronomR_', 'that', 'which', 'who', 'whom', 'whose'], ['CONJ_', 'and', 'or', 'but', 'so', 'yet', 'nor', 'although', 'because', 'since', 'while', 'when', 'if', 'though', 'unless'], ['MOD', 'am', 'is', 'are', 'was', 'were', 'have', 'has', 'had', 'will', 'would', 'can', 'could', 'shall', 'should', 'may', 'might', 'must', 'do', 'does', 'did'], ['PUNCT_', '.', '"', '"', ':', ';', '?', '!', ',', "'", "'", '“', '”', '’']]
[['1', 'DET+?', 'chkNom'], ['2', 'DET+DET', 'chkVerb'], ['3', 'DET+PREP', 'chkVerb'], ['4', 'PREP+?', 'chkPrep'

## Chunkeur

In [6]:
def chunker(tokens, lexique, regles):

    chunks = []
    chk_courant = []
    nums=[]
    cats=[]

    i = 0

    while i < len(tokens): # Parcours des tokens

        tok = tokens[i]
        cat1 = []
        cat2 = []
        
        # Rechercher de la catégorie du token courant "cat1"
        found_lexP1 = False
        for lexP1 in lexique: 
            if tok.lower() in lexP1: # Si catégorie trouvée pour le token courant
                found_lexP1 = True
                cat1.append(lexP1[0]) # L'élément à l'indice 0 est la catégorie selon le fichier lexique
                
                if cat1[0][-1] == "_" : # Pour les mots qui constistuent a eux seuls un chunk, ex.:PUNCT, CONJ, PronomR...
                    
                    regle = f"{cat1[0]}"
                    
                    found_regle = False
                    # On cherche la règle qui s'applique en fonction des deux catégories
                    for elmt in regles:
                        if regle in elmt: # si règle trouvée
                            found_regle = True
                            if len(chk_courant)>0: # Pour évirer d'ajouter un liste vide
                                chunks.append(chk_courant) # On rajoute le chunk courant à liste de chunks afin de le réinitialiser pour ouvrir un nouveau chunk
                                if len(cats)<len(chunks):# Pour les chunks sans catégorie
                                    nums.append("0")
                                    cats.append("None") 
                                
                                    
                            nums.append(elmt[0]) # L'élément à l'indice 0 est le numero de la règle selon le fichier des règles
                            cats.append(elmt[-1]) # L'élément à l'indice -1 est le catégorie du chunk selon le fichier des règles
                            chk_courant = [] # réinitialisation et ouverture d'un nouveau chunk
                            chk_courant.append(tok)
                            chunks.append(chk_courant)
                            chk_courant = []
                            i += 1 # avancer de 1
                            break
                    
                    if not found_regle: # si règle non trouvée
                        chk_courant.append(tok)
                        i += 1
                
                else :
                    # Rechercher de la catégorie du token suivant "cat2"
                    if i < len(tokens) - 1: # Vérifier si on est pas sur le dernier token du texte
                        found_lexP2 = False
                        for lexP2 in lexique:
                            if tokens[i+1].lower() in lexP2: # Si catégorie trouvée pour le token suivant
                                found_lexP2 = True
                                
                                if lexP2[0][-1] == "_": # s'il s'agit de la catégorie des tokens qui constistuent a eux seuls un chunk, ex.:PUNCT, CONJ, PronomR... on ajoute le token courant au chunk courant
                                    chk_courant.append(tok)
                                    i += 1
                                    break 
                                else:
                                    cat2.append(lexP2[0]) # L'élément à l'indice 0 est la catégorie selon le fichier lexique
                                    
                                    # On cherche la règle qui s'applique en fonction des deux catégories
                                    regle = f"{cat1[0]}+{cat2[0]}"
                                
                                    found_regle = False
                                    for elmt in regles: 
                                        if regle in elmt:
                                            found_regle = True
                                            if len(chk_courant)>0: # Pour évirer d'ajouter un liste vide
                                                chunks.append(chk_courant) # On rajoute le chunk courant à liste de chunks afin de le réinitialiser pour ouvrir un nouveau chunk
                                                if len(cats)<len(chunks):# Pour les chunks sans catégorie
                                                    nums.append("0")
                                                    cats.append("None") 
                                                    
                                            nums.append(elmt[0]) # L'élément à l'indice 0 est le numero de la règle selon le fichier des règles
                                            cats.append(elmt[-1]) # L'élément à l'indice -1 est le catégorie du chunk selon le fichier des règles
                                            
                                            chk_courant = [] # réinitialisation et ouverture d'un nouveau chunk
                                            chk_courant.append(tok)
                                            chk_courant.append(tokens[i+1])
                                            i += 2  # Saute de 2 
                                            break
                                    
                                    if not found_regle: # si règle non trouvée
                                        chk_courant.append(tok)
                                        i += 1
                                    break
                                                
                        if not found_lexP2:  # Si aucune catégorie trouvée pour le token i+1
                            cat2.append("?")
                        
                            # On cherche la règle qui s'applique en fonction des deux catégories
                            regle = f"{cat1[0]}+{cat2[0]}"
                        
                            found_regle = False
                            for elmt in regles: 
                                if regle in elmt:
                                    found_regle = True
                                    if len(chk_courant)>0: # Pour évirer d'ajouter un liste vide
                                        chunks.append(chk_courant) # On rajoute le chunk courant à liste de chunks afin de le réinitialiser pour ouvrir un nouveau chunk
                                        if len(cats)<len(chunks):# Pour les chunks sans catégorie
                                            nums.append("0")
                                            cats.append("None") 
                                            
                                    nums.append(elmt[0]) # L'élément à l'indice 0 est le numero de la règle selon le fichier des règles
                                    cats.append(elmt[-1]) # L'élément à l'indice -1 est le catégorie du chunk selon le fichier des règles
                                            
                                    chk_courant = [] # réinitialisation et ouverture d'un nouveau chunk
                                    chk_courant.append(tok)
                                    chk_courant.append(tokens[i+1])
                                    i += 2  # Saute de 2 
                                    break
                            
                            if not found_regle: # si règle non trouvée
                                chk_courant.append(tok)
                                i += 1
                    else:
                        chk_courant.append(tok)
                        i += 1
                        
                        if i == len(tokens) and len(chk_courant)>0: # Pour ne pas omettre le dernier chunk du texte
                            if len(chk_courant)>0:
                                chunks.append(chk_courant) # On rajoute le chunk courant à liste de chunks afin de le réinitialiser pour ouvrir un nouveau chunk
                                if len(cats)<len(chunks):# Pour les chunks sans catégorie
                                    nums.append("0")
                                    cats.append("None")
                                
                break
                                                        
        if not found_lexP1:  # Si catégorie non trouvée pour le token courant
            if tok[0].isupper(): #Pour les noms propres
                regle = "Maj"
                found_regle = False
                for elmt in regles: 
                    if regle in elmt:
                        found_regle = True
                        if len(chk_courant)>0: # Pour ne pas omettre le dernier chunk du texte
                            chunks.append(chk_courant) # On rajoute le chunk courant à liste de chunks afin de le réinitialiser pour ouvrir un nouveau chunk
                            if len(cats)<len(chunks):# Pour les chunks sans catégorie
                                nums.append("0")
                                cats.append("None") 
                            
                        nums.append(elmt[0]) # L'élément à l'indice 0 est le numero de la règle selon le fichier des règles
                        cats.append(elmt[-1]) # L'élément à l'indice -1 est le catégorie du chunk selon le fichier des règles
        
                        chk_courant = [] # réinitialisation et ouverture d'un nouveau chunk
                        chk_courant.append(tok)
                        i += 1 
                        break
                
                if not found_regle: # si règle non trouvée
                    chk_courant.append(tok)
                    i += 1

            else:
                chk_courant.append(tok)
                i += 1  # avancer de 1  
                
                if i == len(tokens) and len(chk_courant)>0: # Pour ne pas omettre le dernier chunk du texte
                    if len(chk_courant)>0: 
                        chunks.append(chk_courant) # On rajoute le chunk courant à liste de chunks afin de le réinitialiser pour ouvrir un nouveau chunk
                        if len(cats)<len(chunks):# Pour les chunks sans catégorie
                            nums.append("0")
                            cats.append("None")
                    
                    

    # Écrire les chunks dans un fichier "Chunks.xml"

    with open("Chunks_en.xml", mode='w', encoding ='utf8') as f:
        xml_deb = "<?xml version=\"1.0\" encoding=\"UTF-8\"?>\n""<annotation>\n\t<texte>\n"
        xml_fin = "\t</texte>\n</annotation>"
        f.write(xml_deb)
        a = 0
        for a in range (len(chunks)):
            num = nums[a]
            cat = cats[a]
            chk = chunks[a]
            
            elmts=""
            for elmt in chk:
                if elmt != chk[-1]: # Pas d'espace après le dernier élément
                    elmts += elmt + " "
                else :
                    elmts += elmt
                
            xml_line = f"\t\t<CHK num_rgl='{num}' cat_chk='{cat}'>{elmts}</CHK>\n"
            # print(xml_line)
            f.write(xml_line)
        f.write(xml_fin)

In [7]:
chunker(tokens, lexique, regles)